In [ ]:
!pip install -q langchain langchain-community langchain-openai faiss-cpu python-dotenv tiktoken

In [ ]:
import os
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
load_dotenv()

FIREWORKS_API_KEY = os.getenv("FIREWORKS_API_KEY")
FIREWORKS_BASE_URL = "https://api.fireworks.ai/inference/v1"

CHAT_MODEL = "accounts/fireworks/models/gpt-oss-20b"
EMBED_MODEL = "accounts/fireworks/models/qwen3-embedding-4b"

In [ ]:
embeddings = OpenAIEmbeddings(
    model=EMBED_MODEL,
    api_key=FIREWORKS_API_KEY,
    base_url=FIREWORKS_BASE_URL,
)

In [ ]:
loader = TextLoader("sample_knowledge.txt", encoding="utf-8")
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)
print(f"Number of chunks: {len(chunks)}")

In [ ]:
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [ ]:
llm = ChatOpenAI(
    model=CHAT_MODEL,
    api_key=FIREWORKS_API_KEY,
    base_url=FIREWORKS_BASE_URL,
    temperature=0,
)

In [ ]:
prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant.
Answer the user's question only using the context below.
If the answer is not in the context, say you do not know.

Context:
{context}

Question:
{question}
""")

In [ ]:
def ask_rag(question: str):
    retrieved_docs = retriever.invoke(question)
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])

    messages = prompt.format_messages(context=context, question=question)
    response = llm.invoke(messages)

    print("QUESTION:\n")
    print(question)
    print("\nRETRIEVED CONTEXT:\n")
    print(context)
    print("\nANSWER:\n")
    print(response.content)

In [ ]:
ask_rag("What is the difference between serverless and dedicated endpoints?")

In [ ]:
ask_rag("Why are token throughput and latency important in user-facing applications?")